# PHẦN 3: THUẬT TOÁN FLAJOLET-MARTIN CẢI TIẾN (128 HASH + MEDIAN OF MEANS)
## Chuyên đề Xử Lý Dữ Liệu Lớn - Giảng viên: Trần Thị Nhi

### 1. Cơ sở lý thuyết & Kỹ thuật Giảm Sai số
- **Vấn đề của FM đơn lẻ:** Một bộ đếm FM 1 hash có độ lệch chuẩn $\sigma(R) \approx 1.12$ bits, tạo ra sai số tương đối lên tới $\approx 78\%$.
- **Giải pháp phối hợp hai tầng (Median-of-Means):**
  1. **Khởi tạo $k = 128$ hàm băm độc lập:** Thay vì chỉ dùng 1 hàm băm, hệ thống sử dụng 128 hàm băm với seed khác nhau (`seed = i * 100 + 7`).
  2. **Chia thành $g = 16$ nhóm độc lập:** Mỗi nhóm quản lý $\frac{128}{16} = 8$ hàm băm.
  3. **Tầng 1 - Tính Trung bình (Mean) trong từng nhóm:**
     - Với mỗi nhóm $g$, tính trung bình số lượng bit 0 lớn nhất: $\bar{r} = \frac{1}{8} \sum_{i=1}^8 r_i$.
     - Ước lượng của nhóm được tính bằng cách mũ hóa: $\hat{n}_g = \frac{2^{\bar{r}}}{\phi}$ với $\phi \approx 0.77351$.
     - *Tác dụng:* Làm mượt đường ước lượng, biến bước nhảy từ số mũ rời rạc thành giá trị liên tục, giảm phương sai cục bộ.
  4. **Tầng 2 - Lấy Trung vị (Median) giữa các nhóm:**
     - Sắp xếp 16 giá trị ước lượng của 16 nhóm và lấy giá trị trung vị ở giữa.
     - *Tác dụng:* Dựa trên bất đẳng thức tập trung xác suất (Chernoff/Chebyshev Bounds), phép lấy trung vị giúp loại bỏ hoàn toàn các giá trị ngoại lai (outliers) do một số hàm băm ngẫu nhiên sinh ra quá nhiều bit 0 bất thường.
- **Đặc tính kỹ thuật:**
  - **Thời gian:** $O(k)$ cho mỗi phần tử ($k = 128$ phép băm).
  - **Bộ nhớ:** $O(k)$ - Mảng 128 số nguyên (khoảng 1.2 KB RAM), vẫn **nhỏ hơn Set hàng ngàn lần**.
  - **Độ chính xác:** Sai số được kiểm soát chặt chẽ, đồ thị ước lượng bám sát Ground Truth.

In [ ]:
# 1. Khởi tạo và nạp các thư viện cần thiết
import os
import sys
import time
import json
import hashlib
import tracemalloc
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (12, 5)
print("Đã nạp xong thư viện thành công!")

In [ ]:
# 2. Cấu hình đường dẫn dữ liệu (Hỗ trợ cả Local và Google Colab)
def get_log_file_path():
    candidates = [
        "accessLog/access.log",
        "/content/drive/MyDrive/accessLog/access.log",
        "../accessLog/access.log",
        "access.log"
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return "accessLog/access.log"

LOG_FILE = get_log_file_path()
print(f"Đường dẫn file log: {LOG_FILE}")

In [ ]:
# 3. Các hàm băm và đếm bit 0
def count_trailing_zeros(n):
    if n == 0:
        return 32
    zeros = 0
    while (n & 1) == 0:
        zeros += 1
        n >>= 1
    return zeros

def hash_function(data_str, seed):
    salted_input = f"{seed}_{data_str}".encode('utf-8')
    hash_digest = hashlib.md5(salted_input).hexdigest()
    return int(hash_digest[:8], 16)

print("Đã định nghĩa xong hàm băm toán học!")

In [ ]:
# 4. Cài đặt lớp Thuật toán Flajolet-Martin Cải Tiến (Median of Means)
class FlajoletMartinAdvanced:
    """
    Thuật toán Flajolet-Martin cải tiến:
    Sử dụng m hàm băm + Kỹ thuật Median of Means để thu hẹp sai số
    """
    def __init__(self, num_hashes=128, num_groups=16):
        self.num_hashes = num_hashes
        self.num_groups = num_groups
        self.hashes_per_group = num_hashes // num_groups
        self.max_zeros = [0] * num_hashes
        self.phi = 0.77351

    def update(self, item):
        """Băm item qua toàn bộ 128 hàm băm độc lập"""
        for i in range(self.num_hashes):
            hash_val = hash_function(item, seed=i*100 + 7)
            r = count_trailing_zeros(hash_val)
            if r > self.max_zeros[i]:
                self.max_zeros[i] = r

    def estimate(self):
        """Tính toán kết hợp: Mean trong nhóm và Median giữa các nhóm"""
        group_averages = []
        for g in range(self.num_groups):
            start_idx = g * self.hashes_per_group
            end_idx = start_idx + self.hashes_per_group
            group_zeros = self.max_zeros[start_idx:end_idx]

            # 1. Tính trung bình số bit 0 trong nhóm
            avg_r = sum(group_zeros) / len(group_zeros)
            # 2. Mũ hóa để tính ước lượng nhóm
            avg_est = (2 ** avg_r) / self.phi
            group_averages.append(avg_est)

        # 3. Lấy trung vị (Median) của 16 nhóm
        group_averages.sort()
        mid = len(group_averages) // 2
        if len(group_averages) % 2 == 0:
            return (group_averages[mid - 1] + group_averages[mid]) / 2.0
        else:
            return group_averages[mid]

print("Đã cài đặt xong lớp FlajoletMartinAdvanced!")

In [ ]:
# 5. Hàm đọc luồng dữ liệu (Streaming Generator)
def stream_log_file(file_path):
    if not os.path.exists(file_path):
        print(f"[CẢNH BÁO] Không tìm thấy file: {file_path}")
        return
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            line = line.strip()
            if line:
                yield line.split(' ', 1)[0]

In [ ]:
# 6. Hàm thực nghiệm Flajolet-Martin Cải Tiến
def run_fm_advanced_experiment(log_file_path, sample_step=50000, max_lines=None):
    fm_adv = FlajoletMartinAdvanced(num_hashes=128, num_groups=16)
    stream_counts = []
    estimates_history = []

    exact_lookup = {}
    if os.path.exists('results/set_metrics.json'):
        with open('results/set_metrics.json', 'r', encoding='utf-8') as f:
            set_data = json.load(f)
            for s, u in zip(set_data.get('history_steps', []), set_data.get('history_unique', [])):
                exact_lookup[s] = u

    print("="*70)
    print("BẮT ĐẦU THỰC NGHIỆM FLAJOLET-MARTIN CẢI TIẾN (128 HASH - MEDIAN OF MEANS)")
    print(f"File: {log_file_path} | Sample step: {sample_step:,} dòng")
    print("="*70)

    tracemalloc.start()
    start_time = time.time()
    total_processed = 0

    for ip in stream_log_file(log_file_path):
        total_processed += 1
        fm_adv.update(ip)

        if total_processed % sample_step == 0:
            est = fm_adv.estimate()
            stream_counts.append(total_processed)
            estimates_history.append(est)

            exact_str = f"{exact_lookup[total_processed]:,}" if total_processed in exact_lookup else "Chưa có"
            print(f"Dòng: {total_processed:>10,} | Ước lượng FM Cải tiến: {int(est):>10,} | Thực tế: {exact_str}")

        if max_lines and total_processed >= max_lines:
            break

    elapsed_time = time.time() - start_time
    _, peak_ram = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    final_estimate = fm_adv.estimate()
    fm_adv_bytes = sys.getsizeof(fm_adv) + sys.getsizeof(fm_adv.max_zeros) + sum(sys.getsizeof(x) for x in fm_adv.max_zeros)

    print("\n" + "="*70)
    print("KẾT QUẢ TỔNG KẾT (FM CẢI TIẾN 128 HASH):")
    print(f"- Tổng số dòng log đã quét:            {total_processed:,}")
    print(f"- Số IP ước lượng cuối cùng:           {int(final_estimate):,}")
    print(f"- Thời gian thực thi:                  {elapsed_time:.2f} giây")
    print(f"- Dung lượng cấu trúc FM 128 Hash:     {fm_adv_bytes:,} Bytes ({fm_adv_bytes/1024:.2f} KB)")
    print(f"- Peak RAM hệ thống:                   {peak_ram / (1024*1024):.4f} MB")
    print("="*70)

    # Lưu kết quả sang results/fm_advanced_metrics.json
    os.makedirs('results', exist_ok=True)
    results_data = {
        "method": "Flajolet-Martin Advanced (128 Hash - Median of Means)",
        "total_processed": total_processed,
        "num_hashes": 128,
        "num_groups": 16,
        "estimate_final": round(final_estimate, 2),
        "elapsed_time": round(elapsed_time, 2),
        "fm_bytes": fm_adv_bytes,
        "peak_ram_bytes": peak_ram,
        "history_steps": stream_counts,
        "history_estimates": [round(e, 2) for e in estimates_history]
    }
    with open('results/fm_advanced_metrics.json', 'w', encoding='utf-8') as f:
        json.dump(results_data, f, indent=2, ensure_ascii=False)
    print("-> Đã lưu kết quả thành công vào 'results/fm_advanced_metrics.json'!")

    return stream_counts, estimates_history

In [ ]:
# 7. Chạy thực nghiệm
steps, adv_estimates = run_fm_advanced_experiment(LOG_FILE, sample_step=50000)

In [ ]:
# 8. Trực quan hóa kết quả FM Cải tiến
if steps:
    plt.figure(figsize=(10, 5))
    plt.plot(steps, adv_estimates, 'g-', linewidth=2, label='FM Cải tiến (128 Hash - Median of Means)')
    plt.title('Đường Ước Lượng của Thuật toán Flajolet-Martin Cải Tiến', fontsize=12, fontweight='bold')
    plt.xlabel('Số dòng log đã xử lý')
    plt.ylabel('Số lượng IP ước lượng')
    plt.grid(True, linestyle=':', alpha=0.6)
    plt.legend()
    plt.tight_layout()
    plt.show()

### 9. Nhận xét và Đánh giá FM Cải Tiến
- **Độ mượt mà và ổn định:** Nhờ phép tính trung bình trong nhóm kết hợp trung vị giữa các nhóm, đường ước lượng không còn bị nhảy bậc thô bạo theo lũy thừa của 2 như FM 1 hash mà tăng trưởng mượt mà, bám rất sát xu hướng tăng trưởng của tập dữ liệu thực tế.
- **Hiệu quả bộ nhớ:** Cấu trúc 128 hash chỉ tốn khoảng $1.2$ KB RAM, hoàn toàn không đáng kể so với tài nguyên hệ thống hiện đại.
- **Chi phí thời gian:** Do phải tính 128 hàm băm cho mỗi dòng log, thời gian CPU xử lý sẽ tăng lên so với FM cơ bản (đây chính là sự đánh đổi Trade-off giữa thời gian tính toán và độ chính xác).